# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad-imran2891/week1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/saad-imran2891/week1.git /content/week1

import sys, subprocess
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

REPO_ROOT = Path('/content/week1')
assert (REPO_ROOT / 'scripts' / 'ml_utils.py').exists(), "Clone failed or repo layout differs — check the git clone output above."

SCRIPTS_DIR = REPO_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from ml_utils import PROCESSED_DIR, OUTPUT_DIR, RAW_PATH, CHART_DIR, read_json

!pip install duckdb huggingface_hub -q

Cloning into '/content/week1'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 143 (delta 53), reused 99 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 13.72 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [2]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}');")

WAREHOUSE_BASE = "hf://datasets/FlyRank/internship-warehouse"

## 1. Question

*The research question and the decision it supports.*
Answer:Question: Given trailing search and engagement performance for existing content, which
pages should be prioritized for refresh review this cycle, and why — across the full
client base, not just the 32-client teaching sample?

Decision this supports: a content team with limited review capacity per sprint needs a
ranked, reason-coded queue rather than reviewing every page. The reference pipeline
already validates this approach on a 30k-page sample (baseline Precision@50 ≈0.24,
model 0.68–0.74). This capstone tests whether that lift holds at full warehouse scale,
across all 104 clients, not just the ones in the bundled slice.

Lane: Refresh / Content Opportunity Scoring (Lane 2).

In [ ]:
COLUMN_MAP = {
    'report_date': 'report_date',
    'content_id': 'content_hash_id',
    'client_id': 'client_hash_id',
    'impressions': 'gsc_impressions',
    'clicks': 'gsc_clicks',
    'position_sum': 'gsc_sum_position',
    'sessions': 'ga4_sessions',
    'pageviews': 'ga4_pageviews',
    'users': 'ga4_users',
    'engaged_sessions': 'ga4_engaged_sessions',
    'ai_sessions': 'sessions_ai',
    'scroll_events': 'scroll_events',
    'gsc_data_available': 'gsc_data_available',
    'ga4_data_available': 'ga4_data_available',
}
cm = COLUMN_MAP

agg_query = f"""
WITH cutoff AS (
    SELECT MAX({cm['report_date']}) AS max_date
    FROM read_parquet('{WAREHOUSE_BASE}/fact_content_daily_performance/**/*.parquet')
),
windowed AS (
    SELECT f.*, cl.gsc_data_start, c.max_date
    FROM read_parquet('{WAREHOUSE_BASE}/fact_content_daily_performance/**/*.parquet') f
    JOIN read_parquet('{WAREHOUSE_BASE}/dim_clients.parquet') cl
        ON f.{cm['client_id']} = cl.client_hash_id
    CROSS JOIN cutoff c
    WHERE f.{cm['report_date']} > c.max_date - INTERVAL 90 DAY
      AND (cl.gsc_data_start IS NULL OR f.{cm['report_date']} >= cl.gsc_data_start)
)
SELECT
    {cm['content_id']} AS content_id,
    {cm['client_id']} AS client_id,
    SUM({cm['impressions']}) AS impressions_90d,
    SUM({cm['clicks']}) AS clicks_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['pageviews']} ELSE 0 END) AS pageviews_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['sessions']} ELSE 0 END) AS sessions_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['users']} ELSE 0 END) AS users_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['engaged_sessions']} ELSE 0 END) AS engaged_sessions_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['ai_sessions']} ELSE 0 END) AS ai_sessions_90d,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE THEN {cm['scroll_events']} ELSE 0 END) AS scroll_events_90d,
    SUM(CASE WHEN {cm['gsc_data_available']} IS TRUE AND {cm['impressions']} > 0 THEN 1 ELSE 0 END) AS days_with_impressions,
    SUM(CASE WHEN {cm['ga4_data_available']} IS TRUE AND {cm['sessions']} > 0 THEN 1 ELSE 0 END) AS days_with_sessions,
    CASE WHEN SUM(CASE WHEN {cm['gsc_data_available']} IS TRUE THEN {cm['impressions']} ELSE 0 END) = 0 THEN 0
         ELSE SUM(CASE WHEN {cm['gsc_data_available']} IS TRUE THEN {cm['position_sum']} ELSE 0 END)
              / SUM(CASE WHEN {cm['gsc_data_available']} IS TRUE THEN {cm['impressions']} ELSE 0 END)
    END AS avg_position,
    SUM(CASE WHEN {cm['report_date']} > max_date - INTERVAL 30 DAY THEN {cm['impressions']} ELSE 0 END) AS impressions_last_30d,
    SUM(CASE WHEN {cm['report_date']} > max_date - INTERVAL 30 DAY THEN {cm['clicks']} ELSE 0 END) AS clicks_last_30d,
    SUM(CASE WHEN {cm['report_date']} > max_date - INTERVAL 30 DAY AND {cm['ga4_data_available']} IS TRUE THEN {cm['sessions']} ELSE 0 END) AS sessions_last_30d,
    SUM(CASE WHEN {cm['report_date']} <= max_date - INTERVAL 30 DAY THEN {cm['impressions']} ELSE 0 END) AS impressions_prev_30d,
    SUM(CASE WHEN {cm['report_date']} <= max_date - INTERVAL 30 DAY THEN {cm['clicks']} ELSE 0 END) AS clicks_prev_30d,
    SUM(CASE WHEN {cm['report_date']} <= max_date - INTERVAL 30 DAY AND {cm['ga4_data_available']} IS TRUE THEN {cm['sessions']} ELSE 0 END) AS sessions_prev_30d
FROM windowed
GROUP BY 1, 2
"""

rollup = con.execute(agg_query).df()
print(rollup.shape)
rollup.head()

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*
Answer:- Release: FlyRank internship warehouse (Hugging Face, gated) — dim_clients.parquet,
  dim_content.parquet, fact_content_daily_performance/ (partitioned by month)
- Not used: fact_content_query_90d.parquet — the model's feature set doesn't consume
  query-mix features, and that table's fixed 90-day window creates a leakage risk
  against a trailing-window label. Left out rather than assumed safe.
- Window: trailing 90 days ending at the warehouse's global MAX(report_date), split
  into last-30d/prev-30d — mirrors how content_refresh_anonymized.csv was built.
- Per-client filter: only report_dates on/after each client's own gsc_data_start
  (unbalanced panel — clients have different history depths; gsc_data_start can
  itself be NULL for some clients, handled as "no start restriction" rather than
  excluding them).
- GA4/GSC filters: fact_content_daily_performance has BOTH gsc_data_available and
  ga4_data_available flags — each metric family summed only where its own flag
  IS TRUE (not = TRUE, since these can be NULL, not just FALSE).
- avg_position: computed as SUM(gsc_sum_position) / SUM(gsc_impressions) over days
  where gsc_data_available IS TRUE — the raw table stores a position SUM per day, not
  a pre-averaged value like the bundled CSV.
- AI traffic: taken from sessions_ai (a rollup column); per-platform breakdowns
  (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other)
  exist but are unused here.
- content_age_days / days_since_last_update: derived as date differences from
  dim_content's content_created_date and last_optimized_date to the snapshot's max
  report_date, since the warehouse doesn't ship these as precomputed ages. Using
  last_optimized_date as "last update" is an assumption worth stating explicitly.
- Exclusions: impressions_90d > 0 and content_age_days >= 90, same as the reference
  pipeline.
- Public-safe: only pseudonymous content_hash_id/client_hash_id used; no domains,
  URLs, or client names anywhere.

In [ ]:
content_query = f"""
WITH cutoff AS (
    SELECT MAX(report_date) AS max_date
    FROM read_parquet('{WAREHOUSE_BASE}/fact_content_daily_performance/**/*.parquet')
)
SELECT
    content_hash_id AS content_id,
    search_volume, competition, competition_level, cpc, content_type, main_intent,
    word_count, char_count, provider_used, model_used,
    date_diff('day', content_created_date, (SELECT max_date FROM cutoff)) AS content_age_days,
    date_diff('day', last_optimized_date, (SELECT max_date FROM cutoff)) AS days_since_last_update
FROM read_parquet('{WAREHOUSE_BASE}/dim_content.parquet')
"""
content_meta = con.execute(content_query).df()
merged = rollup.merge(content_meta, on='content_id', how='left')

merged['ctr'] = np.where(merged['impressions_90d'] > 0, merged['clicks_90d'] / merged['impressions_90d'] * 100, 0).round(2)
merged['engagement_rate'] = np.where(merged['sessions_90d'] > 0, merged['engaged_sessions_90d'] / merged['sessions_90d'] * 100, 0)
merged['scroll_rate'] = np.where(merged['pageviews_90d'] > 0, merged['scroll_events_90d'] / merged['pageviews_90d'] * 100, np.nan)
merged['ai_traffic_pct'] = np.where(merged['sessions_90d'] > 0, merged['ai_sessions_90d'] / merged['sessions_90d'] * 100, 0)
merged['trend_pct'] = np.where(
    merged['impressions_prev_30d'] > 0,
    (merged['impressions_last_30d'] - merged['impressions_prev_30d']) / merged['impressions_prev_30d'] * 100,
    np.nan,
).round(1)

def trend_direction(row):
    last, prev = row['impressions_last_30d'], row['impressions_prev_30d']
    if prev == 0 and last > 0: return 'new'
    if prev == 0 and last == 0: return 'flat'
    pct = (last - prev) / prev * 100
    if pct > 20: return 'up'
    if pct < -20: return 'down'
    return 'stable'

merged['trend_direction'] = merged.apply(trend_direction, axis=1)

merged['age_tier'] = pd.cut(merged['content_age_days'], bins=[-1,14,30,90,180,365,10**9],
                              labels=['0-14','15-30','31-90','91-180','181-365','365+'])
merged['freshness_tier'] = pd.cut(merged['days_since_last_update'], bins=[-1,30,90,180,10**9],
                                    labels=['0-30','31-90','91-180','181+'])
merged['word_count_tier'] = pd.cut(merged['word_count'], bins=[-1,1000,2000,3500,10**9],
                                     labels=['<1000','1000-2000','2000-3500','3500+'])
merged['impression_tier'] = pd.cut(merged['impressions_90d'], bins=[-1,0,300,3000,30000,10**12],
                                     labels=['none','low','moderate','good','excellent'])

def position_tier(pos):
    if pos == 0: return 'no_data'
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

merged['position_tier'] = merged['avg_position'].apply(position_tier)

warehouse_raw_path = RAW_PATH.parent / 'content_refresh_warehouse.csv'
merged.to_csv(warehouse_raw_path, index=False)
print(f"Wrote {len(merged):,} rows to {warehouse_raw_path}")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*
Answer:This capstone runs the repo's existing, already-leakage-checked pipeline against the
warehouse-scale rollup built above, unmodified:

- Label: is_declining_label = (trend_direction == 'down'), built in
  01_prepare_features.py.
- Leakage check: trend_direction and trend_pct are excluded from both
  MODEL_NUMERIC_FEATURES and MODEL_CATEGORICAL_FEATURES in ml_utils.py — they're the
  label source, never features.
- Baseline: 02_baseline_score.py's transparent hand-rule score (0.40×visibility +
  0.30×freshness risk + 0.25×position opportunity + 0.05×depth gap), no training.
- Model: 03_train_model.py trains logistic regression, decision tree, and random
  forest, selects the best by precision_at_50 on a client-holdout split (entire
  clients held out, not random rows).
- Assumption carried over: content_age_days >= 90 and impressions_90d > 0 filters,
  same as the bundled sample.
- Real schema confirmed via DESCRIBE: client_hash_id/content_hash_id (not the
  generic IDs I first assumed), separate gsc_data_available/ga4_data_available flags,
  gsc_sum_position as a per-day SUM (divided by impressions here to get an average).
- Not reused: fact_content_query_90d.parquet — out of scope since the model doesn't
  consume query-mix features.

In [ ]:
WH_FEATURES = PROCESSED_DIR / 'refresh_feature_vector_warehouse.csv'
WH_BASELINE = PROCESSED_DIR / 'baseline_refresh_queue_warehouse.csv'
WH_PREDICTIONS = PROCESSED_DIR / 'model_predictions_warehouse.csv'
WH_RESULTS = OUTPUT_DIR / 'model_results_warehouse.json'
WH_QUEUE = OUTPUT_DIR / 'refresh_queue_warehouse.csv'
WH_REPORT = OUTPUT_DIR / 'model_report_warehouse.md'

def run(script, args):
    result = subprocess.run([sys.executable, str(SCRIPTS_DIR / script)] + args,
                             capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{script} failed")

run('01_prepare_features.py', ['--input', str(warehouse_raw_path), '--output', str(WH_FEATURES)])
run('02_baseline_score.py',   ['--input', str(WH_FEATURES), '--output', str(WH_BASELINE)])
run('03_train_model.py',      ['--features', str(WH_FEATURES), '--baseline', str(WH_BASELINE),
                                 '--predictions', str(WH_PREDICTIONS), '--results', str(WH_RESULTS)])
run('04_evaluate_and_export.py', ['--features', str(WH_FEATURES), '--baseline', str(WH_BASELINE),
                                    '--predictions', str(WH_PREDICTIONS), '--model-results', str(WH_RESULTS),
                                    '--queue', str(WH_QUEUE), '--report', str(WH_REPORT)])

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
wh_results = read_json(WH_RESULTS)

print(f"Rows: {wh_results['input_rows']:,} (train {wh_results['train_rows']:,} / test {wh_results['test_rows']:,})")
print(f"Split strategy: {wh_results['split_strategy']}")
print(f"Target positive rate (declining): {wh_results['target_positive_rate']:.3f}\n")

print(f"{'model':<22}{'roc_auc':>10}{'avg_prec':>10}{'prec@50':>10}{'recall':>10}{'f1':>10}")
for name, m in wh_results['models'].items():
    print(f"{name:<22}{m['roc_auc']:>10.3f}{m['average_precision']:>10.3f}{m['precision_at_50']:>10.3f}{m['recall']:>10.3f}{m['f1']:>10.3f}")
b = wh_results['baseline']
print(f"{'baseline_rules':<22}{b['baseline_roc_auc']:>10.3f}{b['baseline_average_precision']:>10.3f}{b['baseline_precision_at_50']:>10.3f}{'-':>10}{'-':>10}")
print(f"\nBest model: {wh_results['best_model']['name']} (selected by precision_at_50)")

## 5. Limitations

*What this work cannot claim.*
Answer:- Directional, not causal. The model finds pages whose trailing-window features are
  associated with a declining trend label; it says nothing about why, or about
  Google's ranking algorithm.
- Unbalanced panel. Client history depth varies (gsc_data_start/ga4_data_start differ
  per client, and some clients have NULL access flags entirely — access_profile
  "source_only_missing_client_dimension" in dim_clients).
  [Fill in: how many clients/rows were effectively excluded by the 90-day filter.]
- avg_position tier caution. A top_3 page with ~50 impressions/90d is not comparable
  to one with 30,000 — tier medians need a stated volume floor.
  [Fill in: whether your position_tier distribution shows this skew.]
- days_since_last_update is derived from dim_content's last_optimized_date, not a
  direct "last updated" field — an assumption, not a verified equivalence.
- Decision-support only. Use the ranked queue as a reviewer aid, inspect
  high-confidence rows manually — not an auto-publish signal.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
queue = pd.read_csv(WH_QUEUE)
print(queue['suggested_action'].value_counts())
print()
print(queue['confidence'].value_counts())
print()
queue[['final_rank','final_refresh_score','confidence','suggested_action','final_reason_codes']].head(15)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
from IPython.display import SVG, display
for chart_name in ['action_mix.svg', 'confidence_mix.svg', 'top_reason_codes.svg',
                    'top_feature_importance.svg', 'trend_distribution.svg']:
    path = CHART_DIR / chart_name
    if path.exists():
        print(chart_name)
        display(SVG(filename=str(path)))

print(f"\nFull report written to: {WH_REPORT}")
print("Embed this report's tables and these SVGs directly in the deployed paper's Results/Recommendations sections.")

5-minute demo outline
1. (30s) The question — which pages need a refresh, and why, across the full client base.
2. (60s) The data — 79M-row warehouse, 90-day trailing windows, per-client history
   handled, gsc/ga4-availability flags respected.
3. (90s) The baseline vs. model — show the results table live: baseline Precision@50
   vs. best model's Precision@50. State both numbers out loud.
4. (60s) The ranked queue — show the top 10 rows, walk through one refresh reason code
   end to end.
5. (60s) Limitations — directional not causal, decision-support not auto-publish.
6. (20s) Close — link to the deployed paper and repo.

Social-post cut
"Built a content-refresh scoring model on FlyRank's real (anonymized) search warehouse.
A simple hand-rule baseline gets Precision@50 of [X]; a client-holdout-validated model
gets [Y] — meaning the ranked queue actually surfaces the pages worth a human look first.
Full writeup + repro: [link]."

3-sentence employer-facing summary
"I built a ranked content-refresh recommendation system on a ~79M-row real-world search
analytics warehouse, extending an existing reference pipeline (feature engineering →
rule-based baseline → client-holdout-validated ML model → ranked, reason-coded output).
The model improved Precision@50 over the hand-written baseline from [X] to [Y],
validated with a client-level (not row-level) train/test split to prevent leakage across
a client's own pages. I documented data assumptions, leakage checks, and limitations
explicitly, and shipped the full analysis as a public, reproducible research paper."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
